# Week 4 LangGraph Exercise: Improved Sidekick

This notebook **builds on the Sidekick project** in `4_langgraph/` and keeps Week 4 concepts:
- LangGraph `StateGraph` with conditional routing
- tools + tool node
- structured outputs for planner and evaluator
- checkpoint memory with thread IDs
- multi-step loop until success, clarification needed, or retry limit


## 1. Imports and Setup


In [ ]:
import os
import sys
import uuid
import asyncio
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional
from typing_extensions import TypedDict
from typing import Annotated

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode

# Reuse existing sidekick tools from the week folder
sys.path.append(str(Path('4_langgraph').resolve()))
from sidekick_tools import other_tools, playwright_tools

load_dotenv(override=True)

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY is missing in .env')


## 2. Structured Schemas and State


In [ ]:
class PlanOutput(BaseModel):
    plan_steps: List[str] = Field(description='Short actionable plan steps')
    assumptions: List[str] = Field(description='Assumptions made before execution')


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description='Feedback on assistant progress and quality')
    success_criteria_met: bool = Field(description='True if task is complete')
    user_input_needed: bool = Field(description='True if user clarification is required')


class SidekickState(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    plan_steps: List[str]
    assumptions: List[str]
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    attempt_count: int
    max_attempts: int
    final_response: str


## 3. Improved Sidekick Class

Improvements over the base Sidekick:
- explicit **planner node** before work
- explicit retry cap (`max_attempts`) to prevent loops
- cleaner evaluator decisions
- optional Playwright tools toggle


In [ ]:
class ImprovedSidekick:
    def __init__(self, use_playwright: bool = False):
        self.use_playwright = use_playwright
        self.sidekick_id = str(uuid.uuid4())
        self.memory = MemorySaver()

        self.tools = []
        self.browser = None
        self.playwright = None

        self.planner_llm = ChatOpenAI(model='gpt-4o-mini').with_structured_output(PlanOutput)
        self.worker_llm = ChatOpenAI(model='gpt-4o-mini')
        self.evaluator_llm = ChatOpenAI(model='gpt-4o-mini').with_structured_output(EvaluatorOutput)

        self.worker_llm_with_tools = None
        self.graph = None

    async def setup(self):
        self.tools = await other_tools()
        if self.use_playwright:
            p_tools, self.browser, self.playwright = await playwright_tools()
            self.tools += p_tools

        self.worker_llm_with_tools = self.worker_llm.bind_tools(self.tools)
        self._build_graph()

    def _format_conversation(self, messages: List[Any]) -> str:
        lines = []
        for m in messages:
            if isinstance(m, HumanMessage):
                lines.append(f'User: {m.content}')
            elif isinstance(m, AIMessage):
                txt = m.content or '[Tool call]'
                lines.append(f'Assistant: {txt}')
        return '\n'.join(lines)

    def planner_node(self, state: SidekickState) -> Dict[str, Any]:
        user_request = state['messages'][-1].content
        plan_prompt = (
            f'Create a short execution plan for this request: {user_request}\n'
            f'Success criteria: {state["success_criteria"]}\n'
            'Return concise plan steps and assumptions.'
        )
        output = self.planner_llm.invoke(plan_prompt)
        return {
            'plan_steps': output.plan_steps,
            'assumptions': output.assumptions,
        }

    def worker_node(self, state: SidekickState) -> Dict[str, Any]:
        system_prompt = f"""You are an execution assistant that can use tools.
Current datetime: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Goal success criteria:
{state['success_criteria']}

Execution plan:
{state['plan_steps']}

Assumptions:
{state['assumptions']}

Attempts so far: {state['attempt_count']} / {state['max_attempts']}

If you need clarification from user, ask a direct question.
If task is complete, provide final answer clearly.
"""
        if state.get('feedback_on_work'):
            system_prompt += f"""
Evaluator feedback from previous attempt:
{state['feedback_on_work']}
Use this feedback to improve your next response.
"""

        msgs = state['messages']
        has_system = any(isinstance(m, SystemMessage) for m in msgs)
        if not has_system:
            msgs = [SystemMessage(content=system_prompt)] + msgs
        else:
            msgs = [SystemMessage(content=system_prompt) if isinstance(m, SystemMessage) else m for m in msgs]

        response = self.worker_llm_with_tools.invoke(msgs)
        return {
            'messages': [response],
            'attempt_count': state['attempt_count'] + 1,
        }

    def worker_router(self, state: SidekickState) -> str:
        last = state['messages'][-1]
        if hasattr(last, 'tool_calls') and last.tool_calls:
            return 'tools'
        return 'evaluator'

    def evaluator_node(self, state: SidekickState) -> Dict[str, Any]:
        last_response = state['messages'][-1].content
        eval_system = (
            'You are a strict but fair evaluator for task completion. '
            'Decide if success criteria is met, or if user clarification is needed.'
        )
        eval_user = f"""
Conversation so far:
{self._format_conversation(state['messages'])}

Success criteria:
{state['success_criteria']}

Last assistant response:
{last_response}

If the answer is incomplete but improvable without user input, set:
- success_criteria_met=False
- user_input_needed=False
"""
        result = self.evaluator_llm.invoke([
            SystemMessage(content=eval_system),
            HumanMessage(content=eval_user)
        ])

        return {
            'feedback_on_work': result.feedback,
            'success_criteria_met': result.success_criteria_met,
            'user_input_needed': result.user_input_needed,
            'final_response': last_response or '',
        }

    def evaluation_router(self, state: SidekickState) -> str:
        if state['success_criteria_met']:
            return 'END'
        if state['user_input_needed']:
            return 'END'
        if state['attempt_count'] >= state['max_attempts']:
            return 'END'
        return 'worker'

    def _build_graph(self):
        builder = StateGraph(SidekickState)
        builder.add_node('planner', self.planner_node)
        builder.add_node('worker', self.worker_node)
        builder.add_node('tools', ToolNode(tools=self.tools))
        builder.add_node('evaluator', self.evaluator_node)

        builder.add_edge(START, 'planner')
        builder.add_edge('planner', 'worker')

        builder.add_conditional_edges('worker', self.worker_router, {
            'tools': 'tools',
            'evaluator': 'evaluator',
        })
        builder.add_edge('tools', 'worker')

        builder.add_conditional_edges('evaluator', self.evaluation_router, {
            'worker': 'worker',
            'END': END,
        })

        self.graph = builder.compile(checkpointer=self.memory)

    async def run(self, user_message: str, success_criteria: str, max_attempts: int = 4, thread_id: Optional[str] = None):
        tid = thread_id or self.sidekick_id
        config = {'configurable': {'thread_id': tid}}

        state: SidekickState = {
            'messages': [HumanMessage(content=user_message)],
            'success_criteria': success_criteria,
            'plan_steps': [],
            'assumptions': [],
            'feedback_on_work': None,
            'success_criteria_met': False,
            'user_input_needed': False,
            'attempt_count': 0,
            'max_attempts': max_attempts,
            'final_response': '',
        }

        result = await self.graph.ainvoke(state, config=config)
        status = 'completed' if result['success_criteria_met'] else ('needs_user_input' if result['user_input_needed'] else 'stopped_on_retry_limit')

        return {
            'thread_id': tid,
            'status': status,
            'attempt_count': result['attempt_count'],
            'plan_steps': result['plan_steps'],
            'assumptions': result['assumptions'],
            'feedback_on_work': result['feedback_on_work'],
            'final_response': result['final_response'],
        }

    def cleanup(self):
        if self.browser:
            try:
                loop = asyncio.get_running_loop()
                loop.create_task(self.browser.close())
                if self.playwright:
                    loop.create_task(self.playwright.stop())
            except RuntimeError:
                asyncio.run(self.browser.close())
                if self.playwright:
                    asyncio.run(self.playwright.stop())


## 4. Instantiate and Test

Default is `use_playwright=False` for easier notebook execution.


In [ ]:
async def demo_run():
    sidekick = ImprovedSidekick(use_playwright=False)
    await sidekick.setup()

    result = await sidekick.run(
        user_message='Research 3 practical steps to start learning LangGraph and write them to a markdown file in sandbox.',
        success_criteria='Provide 3 concrete steps and ensure the file is written with a useful summary.',
        max_attempts=4,
    )

    print('Status:', result['status'])
    print('Attempts:', result['attempt_count'])
    print('Plan:', result['plan_steps'])
    print('Final response:\n', result['final_response'])
    print('Evaluator feedback:\n', result['feedback_on_work'])

    sidekick.cleanup()

# Run in notebooks
# await demo_run()


## 5. Optional Gradio UI (Single Superstep per click)


In [ ]:
# import gradio as gr
#
# async def setup_app():
#     s = ImprovedSidekick(use_playwright=False)
#     await s.setup()
#     return s
#
# async def process(sidekick, message, criteria, history):
#     out = await sidekick.run(message, criteria, max_attempts=4)
#     history = history + [
#         {'role': 'user', 'content': message},
#         {'role': 'assistant', 'content': out['final_response']},
#         {'role': 'assistant', 'content': f"Evaluator: {out['feedback_on_work']}\nStatus: {out['status']}"},
#     ]
#     return history, sidekick
#
# with gr.Blocks(theme=gr.themes.Default(primary_hue='emerald')) as demo:
#     gr.Markdown('## Improved Sidekick (LangGraph)')
#     sidekick_state = gr.State()
#     chat = gr.Chatbot(type='messages')
#     msg = gr.Textbox(label='Request')
#     criteria = gr.Textbox(label='Success Criteria')
#     run_btn = gr.Button('Run')
#
#     demo.load(setup_app, [], [sidekick_state])
#     run_btn.click(process, [sidekick_state, msg, criteria, chat], [chat, sidekick_state])
#
# demo.launch()
